# Multiscale Hybrid diffusion with Robin local problems

Barrenechea, Gomes and Paredes (2024), [DOI 10.1137/22M1542556](https://doi.org/10.1137/22M1542556). The local operator is coercive Robin, and the global multiplier is $\lambda=(q-p\sigma)\cdot n$, with $q=-K\nabla p$ and $\sigma=\nu(x-a)/2$. It is not the physical normal flux.

This notebook runs a small analytical patch and reads the separate multilevel campaign. The polygonal and triangular mesh connectivity is explicitly chosen for this package; the campaign is not a digitized reproduction of the article figures.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/148a0b5ee337a20784b44f33179e438dd0644cad9aafbe1ae529f3830a82ff02/56_mh-companion.zip"
COMPANION_SHA256 = "148a0b5ee337a20784b44f33179e438dd0644cad9aafbe1ae529f3830a82ff02"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/56_mh.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.fem.traces.interval import FaceSpace, SkeletonSpace
from pymhm.meshes.triangle import TriangleMesh
from pymhm import assemble
from examples.formulations.robin import define_robin, robin_gauge, recover_robin

RESULTS = ROOT / "examples/results/mh"
FIGURES = ROOT / "docs/figures/mh"

In [ ]:
import inspect
from examples.formulations.robin import robin_equations
print(inspect.getsource(robin_equations))
print(inspect.getsource(define_robin))


## Small nonzero Dirichlet patch

For $K=I$, $p=1+x^2+y^2$, the source is $f=-4$ and the physical flux is $q=(-2x,-2y)$. Local P2 and trace P2 represent the Robin multiplier exactly. Testing the local equation with one checks $\int_{\partial K}(\lambda+p\sigma\cdot n)=\int_K f$.

In [ ]:
mesh = TriangleMesh.unit_square(2)
skeleton = SkeletonSpace(mesh, tuple(FaceSpace.uniform(2) for _ in mesh.faces))
exact = lambda x: 1 + np.sum(x**2, axis=1)
with threadpool_limits(1):
    definition = define_robin(mesh, source=-4, dirichlet=exact, skeleton=skeleton, degree=2, local_refinement=2)
    system = assemble(definition.problem)
    coefficients = system.solve(constraints=robin_gauge(definition, system))
    solution = recover_robin(definition, system, coefficients)
pressure_error = solution.l2_error(exact)
flux_error = solution.flux_l2_error(lambda x: -2*x)
balance = float(np.max(abs(solution.conservation_residuals())))
assert max(pressure_error, flux_error, balance) < 1e-10
{"pressure_L2": pressure_error, "physical_flux_L2": flux_error, "macro_balance_max": balance}

## Five-resolution analytical campaign

The published equation has $p=\sin(6\pi x)\sin(14\pi y)$ and $f=232\pi^2p$. Both degree pairs $(\ell,k)=(1,3),(2,4)$ are measured on triangles and on nonconvex L-shaped polygons. The recorded fields use broken, independent interface values. The campaign explicitly retains extended-precision corrections at unchanged residual thresholds; it runs separately from this notebook.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((RESULTS / "comparison.json").read_text())
    assert not record["source_changed_during_run"]
    [(row["mesh"], row["trace_degree"], row["resolution"], row["pressure_relative_error"], row["flux_relative_error"]) for row in record["smooth"]]

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(FIGURES / "convergence.png")))
    display(Image(filename=str(FIGURES / "triangles-fields.png")))
    display(Image(filename=str(FIGURES / "L-polygons-fields.png")))

## The same-space limit as $\nu\to0$

The pressure difference is integrated in the broken energy norm with identical fine meshes, trace spaces, data and quadrature. The reference is the MHM solution in those spaces, not an exact solution. Local and global spectral conditioning is reported separately; no runtime speedup follows from these condition numbers.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record["vanishing_robin"]["rows"]

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    display(Image(filename=str(FIGURES / "vanishing-robin.png")))

The global `Equation` includes the extra boundary-pressure coordinates needed for physical Neumann data. `robin_gauge` fixes the reconstructed pressure mean on a fully Neumann boundary. The local multiplier remains distinct from the physical Darcy outflow.

Generate the corresponding public case data and figures before executing the archived gallery; the reproducible commands are documented in [the Robin method guide](https://ipes-lncc.github.io/pymhm/cases/mh/).
